# 04 — The Model *Is* the Solution

**What "prediction" means for a PINN — demonstrated on a trained checkpoint.**

In normal ML, prediction means generalizing to unseen data. In a PINN, the trained network is a
**continuous approximation of the solution function itself** — `u = model(t)` is the whole
story. This notebook makes that tangible with four demonstrations on a trained
harmonic-oscillator model:

1. **Mesh-free evaluation** — query the solution at any resolution, no re-solving
2. **Derivatives for free** — velocity `u'` and acceleration `u''` via autograd, never trained for
3. **The residual self-check** — validating the solution *without* a reference answer
4. **The extrapolation failure mode** — why the model is only valid inside its trained domain

Full conceptual write-up: [`docs/prediction.md`](../docs/prediction.md).

> **Requirements** — a trained harmonic run must exist (`uv run train-harmonic train`).
> If none is found, the setup cell below trains one (~20 s on CPU).


## 1. Load the Trained Model

Checkpoints are **self-describing**: the run config is stored inside `checkpoint.pt`, so
`load_model` rebuilds the exact architecture automatically — no hyperparameters to remember.
Note what we are *not* doing here: no losses, no optimizer, no collocation points. Solving
already happened, once, at training time.


In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.autograd as autograd
import matplotlib.pyplot as plt

# Run from the repo root so relative paths (outputs/) resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from pinn import set_seed, setup_logging
from experiments.common import find_latest_run, load_model
from experiments.harmonic_oscillator.train import build_model, exact_solution

setup_logging()
set_seed(42)
device = torch.device("cpu")

try:
    run_dir = find_latest_run("harmonic_oscillator")
except FileNotFoundError:
    # No trained run yet — train one now (~20 s on CPU)
    from experiments.harmonic_oscillator.train import solve_harmonic_oscillator
    solve_harmonic_oscillator(show=False)
    run_dir = find_latest_run("harmonic_oscillator")

model, config = load_model(run_dir, build_model, device)
D, W0 = config["d"], config["w0"]
MU, K = 2 * D, W0**2
print(f"Loaded run: {run_dir}")
print(f"Physics: u'' + {MU}*u' + {K}*u = 0   (w0={W0}, d={D})")

## 2. Mesh-Free Evaluation — Any Point, Any Resolution

A classical solver hands you values on a fixed grid; anything in between is interpolation. The
PINN is a *function* — evaluate it on 10,000 points, or at `t = 0.123456789` exactly. The cost
is one forward pass; no physics is re-solved.


In [ ]:
# 10,000-point ultra-smooth evaluation — instant
t_dense = torch.linspace(0, 1, 10_000).view(-1, 1)
with torch.no_grad():
    u_dense = model(t_dense).numpy()

# ... and one arbitrary off-grid point
t_query = torch.tensor([[0.123456789]])
with torch.no_grad():
    u_query = model(t_query).item()
print(f"u(0.123456789) = {u_query:.6f}   (exact: {exact_solution(D, W0, np.array([[0.123456789]]))[0,0]:.6f})")

plt.figure(figsize=(10, 4))
plt.plot(t_dense.numpy(), u_dense, "r-", linewidth=0.8, label="PINN, 10,000 points")
plt.scatter([0.123456789], [u_query], s=80, zorder=5, color="blue", label="arbitrary query point")
plt.xlabel("t"); plt.ylabel("u(t)"); plt.title("Mesh-free evaluation")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 3. Derivatives for Free — Velocity and Acceleration

The network was trained to output **position** `u(t)` only. But because it is differentiable,
autograd extracts physically meaningful derivatives it was never explicitly asked for:

- **velocity** $u'(t)$ — first autograd pass
- **acceleration** $u''(t)$ — second autograd pass

We validate against the exact derivatives. The analytic velocity is

$$u'(t) = e^{-dt}\,2A\big[-d\cos(\phi+\omega t) - \omega\sin(\phi+\omega t)\big]$$

and — elegantly — the exact acceleration comes from the ODE itself:
$u'' = -\mu u' - k u$ (any correct solution must satisfy it).


In [ ]:
t = torch.linspace(0, 1, 500).view(-1, 1).requires_grad_(True)

u = model(t)
u_t = autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]     # velocity
u_tt = autograd.grad(u_t, t, torch.ones_like(u_t), create_graph=True)[0]  # acceleration

t_np = t.detach().numpy()
u_np, u_t_np, u_tt_np = u.detach().numpy(), u_t.detach().numpy(), u_tt.detach().numpy()

# Exact references
w = np.sqrt(W0**2 - D**2)
phi = np.arctan(-D / w)
A = 1 / (2 * np.cos(phi))
u_exact = exact_solution(D, W0, t_np)
u_t_exact = np.exp(-D * t_np) * 2 * A * (-D * np.cos(phi + w * t_np) - w * np.sin(phi + w * t_np))
u_tt_exact = -MU * u_t_exact - K * u_exact   # from the ODE itself

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for ax, (pinn, exact, name) in zip(axes, [
    (u_np, u_exact, "position u(t)  — trained for"),
    (u_t_np, u_t_exact, "velocity u'(t)  — autograd, never trained for"),
    (u_tt_np, u_tt_exact, "acceleration u''(t)  — autograd, never trained for"),
]):
    ax.plot(t_np, exact, "k-", linewidth=2, alpha=0.8, label="exact")
    ax.plot(t_np, pinn, "r--", linewidth=1.5, label="PINN")
    ax.set_ylabel(name.split()[0]); ax.set_title(name); ax.legend(); ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("t")
plt.tight_layout(); plt.show()

for name, pinn, exact in [("u", u_np, u_exact), ("u'", u_t_np, u_t_exact), ("u''", u_tt_np, u_tt_exact)]:
    rel = np.linalg.norm(pinn - exact) / np.linalg.norm(exact)
    print(f"relative L2 error {name:4s}: {rel:.4e}")

Note the error growth pattern: each differentiation amplifies the approximation error
(`u''` scales with $\omega_0^2 = 6400$), yet all three stay accurate — because training
minimised the *residual*, which couples all three derivatives together.

## 4. The Residual Self-Check — Validation Without an Answer Key

Plug the model back into the ODE: $r(t) = u'' + \mu u' + k u$. For the true solution this is
**identically zero**. The pointwise residual magnitude is therefore an *intrinsic* quality map —
it tells you where the solution is trustworthy **without needing the exact solution at all**.

This is the tool that matters for real problems: Burgers and Schrödinger have no closed form,
so the residual check is the *only* pointwise validation available.


In [ ]:
residual = (u_tt + MU * u_t + K * u).detach().numpy()

# Scale-aware comparison: the ODE's individual terms are O(k) = O(6400)
term_scale = np.abs(K * u_np)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(t_np, np.abs(residual) + 1e-12, "b-", linewidth=1)
axes[0].set(title="|residual(t)| = |u'' + mu*u' + k*u|", xlabel="t", ylabel="|r(t)|  (log)")
axes[0].grid(True, alpha=0.3)

rel_residual = np.abs(residual) / (term_scale + 1e-12)
axes[1].semilogy(t_np, rel_residual + 1e-12, "g-", linewidth=1)
axes[1].set(title="residual relative to the k*u term", xlabel="t", ylabel="|r| / |k*u|  (log)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"max |residual|          : {np.abs(residual).max():.4e}")
print(f"mean |residual|         : {np.abs(residual).mean():.4e}")
print(f"scale of k*u term       : {term_scale.max():.1f}")
print(f"worst relative residual : {rel_residual.max():.4e}")

A relative residual of ~$10^{-3}$ or better across the domain means the physics is satisfied
to that precision *everywhere* — the honest, reference-free certificate of solution quality.

## 5. The Failure Mode — Extrapolation Beyond the Trained Domain

Collocation points enforced the physics on $t \in [0, 1]$ **only**. Outside that interval the
network is unconstrained — it outputs *something*, but the physics never shaped it. Watch what
happens on $t \in [0, 2]$:


In [ ]:
t_ext = torch.linspace(0, 2, 1000).view(-1, 1).requires_grad_(True)
u_ext = model(t_ext)
u_ext_t = autograd.grad(u_ext, t_ext, torch.ones_like(u_ext), create_graph=True)[0]
u_ext_tt = autograd.grad(u_ext_t, t_ext, torch.ones_like(u_ext_t), create_graph=True)[0]
residual_ext = (u_ext_tt + MU * u_ext_t + K * u_ext).detach().numpy()

t_ext_np = t_ext.detach().numpy()
u_ext_np = u_ext.detach().numpy()
u_ext_exact = exact_solution(D, W0, t_ext_np)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(t_ext_np, u_ext_exact, "k-", linewidth=1.5, alpha=0.8, label="exact")
axes[0].plot(t_ext_np, u_ext_np, "r--", linewidth=1.5, label="PINN")
axes[0].axvspan(1.0, 2.0, alpha=0.15, color="red", label="untrained region")
axes[0].set(title="Extrapolation: the model diverges outside its trained domain", ylabel="u(t)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(t_ext_np, np.abs(residual_ext) + 1e-12, "b-", linewidth=1)
axes[1].axvspan(1.0, 2.0, alpha=0.15, color="red")
axes[1].set(title="... and the residual check catches it", xlabel="t", ylabel="|r(t)|  (log)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

in_domain = np.abs(residual_ext[t_ext_np[:, 0] <= 1.0]).mean()
out_domain = np.abs(residual_ext[t_ext_np[:, 0] > 1.0]).mean()
print(f"mean |residual| inside  t<=1 : {in_domain:.4e}")
print(f"mean |residual| outside t>1  : {out_domain:.4e}")
print(f"residual blows up by         : {out_domain / in_domain:.1f}x")

The residual explodes the moment we leave the trained domain — which is exactly the point:
**the residual check detects its own region of validity.** You never need to guess where the
model is trustworthy; the physics tells you.

## 6. Takeaways

| Classical ML prediction | PINN prediction |
|---|---|
| Generalize to unseen data from the same distribution | Evaluate a learned *function* at arbitrary points |
| Fixed input → output mapping | Full differentiable solution: `u`, `u'`, `u''`, ... from one set of weights |
| Validation needs held-out labels | Validation is intrinsic: plug the model into the PDE residual |
| May extrapolate plausibly | **Never** extrapolate — physics was only enforced on the collocation domain |

```
train    →  compress the ODE's solution into ~3k weights   (seconds)
predict  →  decompress: forward passes, physics already solved   (milliseconds)
```

The checkpoint is a **compiled solution** to one specific problem instance (`w0=80, d=2` here).
A new `w0` needs a new training run — lifting that is the domain of parametric PINNs and
operator learning (DeepONet, FNO), where the network learns the whole solution *family*.

Full write-up: [`docs/prediction.md`](../docs/prediction.md).
